In [0]:
spark

In [0]:
%sql
 USE CATALOG hive_metastore

In [0]:
%sql
show databases

In [0]:
dbutils.fs.ls('FileStore/tables/')

In [0]:
df_dept = spark.read.csv('dbfs:/FileStore/tables/departments.csv',header ='true', inferSchema='true')


In [0]:
df_dept.display()

In [0]:

# write hive data 
df_dept.write.format('parquet').mode('overwrite').option("path","/data/output/department_parquet_1").saveAsTable('dept_parquet')

In [0]:
#view files
display(dbutils.fs.ls("dbfs:/data/output/department_parquet_1"))

In [0]:
%sql
show tables in default

In [0]:
%sql
desc extended dept_parquet


In [0]:
%sql
SELECT * from dept_parquet where DeptID ='DEPT001'

In [0]:
%sql
-- update dept_parquet set Name = 'Ankh ka department' where DeptID = 'D001';
-- select * from dept_parquet

In [0]:
df_dept.write.format('delta').mode('overwrite').option("path", "/data/output/department_delta_1").saveAsTable('dept_delta')

In [0]:
%sql 
show tables in default

In [0]:
%sql
desc extended dept_delta

In [0]:
display(dbutils.fs.ls("dbfs:/data/output/department_delta_1/_delta_log"))

In [0]:
dbutils.fs.head("dbfs:/data/output/department_delta_1/_delta_log/00000000000000000000.json")

In [0]:
%sql
--#check versio hisiory
describe history dept_delta

In [0]:
%sql
update dept_delta set Name = 'Ankh ka department' where DeptID = 'DEPT001';
select * from dept_delta

In [0]:
%sql
desc history dept_delta

In [0]:
display(dbutils.fs.ls("dbfs:/data/output/department_delta_1/_delta_log"))

In [0]:
#read table 1 as table, 2 as path, check table with version,
#df_delta = spark.read.format('delta').load('dbfs:/data/output/department_delta_1', header = 'true',inferSchema ='true')
#df_delta = spark.read.table('dept_delta')
df_delta = spark.read.table('dept_delta@v3')

df_delta.display()

In [0]:
%sql 
-- #read using sql querires with version

select * from dept_delta@v3 where DeptID ='DEPT001'


In [0]:
%sql
select * from dept_delta@v3 where DeptID ='DEPT001'

### SCHEMA Evolution


In [0]:
%sql
select *, current_timestamp() as time from df_delta@v0 where DeptID = 'DEPT021'

In [0]:
%sql
 SELECT *, current_timestamp() as time
    FROM dept_delta
    VERSION AS OF 3
    WHERE DeptID = 'DEPT020'

In [0]:

df_new =spark.sql("SELECT *, current_timestamp() as time from dept_delta@v0 where DeptID = 'DEPT020'")
df_new.display()


In [0]:
df_new.write.format("delta").mode("append").option("mergeSchema", True).option("path", "dbfs:/data/output/department_delta_1").saveAsTable('dept_delta')

In [0]:
#view data 
display(dbutils.fs.ls('dbfs:/data/output/department_delta_1'))

In [0]:

%sql
--check version 
desc history dept_delta

In [0]:
%sql
select * from dept_delta@v6 where DeptID ='DEPT020'


In [0]:
%sql
-- see time column added to all the records
select * from dept_delta

in case of parquet we have to re write the whole column and but in delta we can add latest schema by adding new row with new schema and merge schema should true.

### reading delta table with delta library

In [0]:
from delta import DeltaTable
dt =DeltaTable.forName(spark, "dept_delta")
dt.history().display()

In [0]:
# convert parquet table into delta table
#check table is delttable or not
DeltaTable.isDeltaTable(spark,"dbfs:/data/output/department_parquet_1")

In [0]:
#convert oto delta 
DeltaTable.convertToDelta(spark,"parquet.`dbfs:/data/output/department_parquet_1`")

In [0]:
DeltaTable.isDeltaTable(spark,"dbfs:/data/output/department_parquet_1")

In [0]:
# validate this location
display(dbutils.fs.ls('dbfs:/data/output/department_parquet_1'))

In [0]:
%sql
-- check meta data 
desc extended dept_parquet

In [0]:
%sql
-- convert meta data to parquet 
Convert to delta dept_parquet

### revert back to particular version

In [0]:
%sql
restore table dept_delta to version as of 1

In [0]:
%sql
desc history dept_delta

In [0]:
%sql
select * from dept_delta@v6

In [0]:
display(dbutils.fs.ls('/data/output/department_delta_1'))
# you will see that there are multiple parquet file with part * to maitain version so we are goint to remove the all version expect latest one

In [0]:
dt = DeltaTable.forName(spark, 'dept_delta')
dt.vacuum(0)
#requirement failed: Are you sure you would like to vacuum files with such a low retention period? If you havewriters that are currently writing to this table, there is a risk that you may corrupt thestate of your Delta table.



In [0]:
# spark.conf.set("spark.delta.retentionDurationCheck.enabled", "false")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
dt = DeltaTable.forName(spark, 'dept_delta')
dt.vacuum(0)


In [0]:
# convertinf delta table to parquet
DeltaTable.forPath(spark, 'dbfs:/data/output/department_parquet_1').toDF().write.parquet('dept_parquet')

In [0]:
%sql 
-- desc extended dept_parquet
convert to parquet dept_parquet
     
